# INCLUDE preprocessing

This notebook loads INCLUDE for Italian, Spanish, Russian, Chinese, and Hindi; cleans and deduplicates the MCQ records; maps `regional_feature` into our `regionality` labels; explores the distribution; and saves SFT-compatible JSONL files under `multilingual/data/processed/include/`.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
REPO_ROOT = Path('/scratch/workspace/emma/standard-project-m2-mystery-machine')
OUTPUT_DIR = REPO_ROOT / 'multilingual/data/processed/include'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'CohereLabs/include-base-44'
LANGUAGE_CONFIGS = {
    'italian': 'italian',
    'spanish': 'spanish',
    'russian': 'russian',
    'chinese': 'chinese',
    'hindi': 'hindi',
}
SPLITS = ['validation', 'test']
ANSWER_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', '0': 'A', '1': 'B', '2': 'C', '3': 'D'}
REGIONALITY_MAP = {
    'culture-related': 'regional',
    'region-explicit': 'regional',
    'region-implicit': 'regional_implicit',
    'agnostic': 'region_agnostic',
}
VALID_ANSWERS = {'A', 'B', 'C', 'D'}
VALID_REGIONALITIES = {'regional', 'regional_implicit', 'region_agnostic'}
HF_TOKEN = os.environ.get('HF_TOKEN')

## Load all languages and splits

In [ ]:
frames = []

for language, config in LANGUAGE_CONFIGS.items():
    for split in SPLITS:
        print(f'Loading {DATASET_NAME} | config={config} | split={split}')
        ds = load_dataset(DATASET_NAME, config, split=split, token=HF_TOKEN)
        df_split = ds.to_pandas()
        df_split['language'] = language
        df_split['source_split'] = split
        df_split['source_idx'] = range(len(df_split))
        frames.append(df_split)

df_raw = pd.concat(frames, ignore_index=True)
print(f'Loaded rows: {len(df_raw)}')
display(df_raw.head())

## Initial exploration

In [ ]:
print('Columns:')
print(df_raw.columns.tolist())

print('
regional_feature values:')
print(sorted(df_raw['regional_feature'].dropna().astype(str).unique().tolist()))

print('
Counts by language:')
display(df_raw['language'].value_counts().sort_index().to_frame('n_examples'))

print('
Counts by regional_feature:')
display(df_raw['regional_feature'].value_counts(dropna=False).to_frame('n_examples'))

print('
Counts by language × regional_feature:')
display(pd.crosstab(df_raw['language'], df_raw['regional_feature'], dropna=False))

print('
Counts by domain:')
display(df_raw['domain'].value_counts(dropna=False).sort_index().to_frame('n_examples'))

print('
Counts by source_split:')
display(pd.crosstab(df_raw['language'], df_raw['source_split']))

In [ ]:
plot_df = pd.crosstab(df_raw['language'], df_raw['regional_feature'], dropna=False)
ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('INCLUDE examples per language and regional_feature')
ax.set_xlabel('Language')
ax.set_ylabel('Number of examples')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Clean and map labels

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()


def map_answer(x):
    if pd.isna(x):
        return None
    if x in ANSWER_MAP:
        return ANSWER_MAP[x]
    sx = str(x).strip().upper()
    return sx if sx in VALID_ANSWERS else ANSWER_MAP.get(sx)


def make_prompt(row):
    return (
        f"{clean_text(row['question'])}

"
        f"A) {clean_text(row['option_a'])}
"
        f"B) {clean_text(row['option_b'])}
"
        f"C) {clean_text(row['option_c'])}
"
        f"D) {clean_text(row['option_d'])}"
    )


df = df_raw.copy()
for col in ['question', 'option_a', 'option_b', 'option_c', 'option_d']:
    df[col] = df[col].map(clean_text)

df['answer'] = df['answer'].map(map_answer)
df['regionality'] = df['regional_feature'].map(REGIONALITY_MAP)

n_before = len(df)
df = df[df['question'].ne('')]
df = df[df['option_a'].ne('') & df['option_b'].ne('') & df['option_c'].ne('') & df['option_d'].ne('')]
df = df[df['answer'].isin(VALID_ANSWERS)]
df = df[df['regionality'].isin(VALID_REGIONALITIES)]

print(f'Rows before basic cleaning: {n_before}')
print(f'Rows after basic cleaning:  {len(df)}')

unknown_regional_features = sorted(set(df_raw['regional_feature'].dropna().astype(str)) - set(REGIONALITY_MAP))
print('Unmapped regional_feature values:', unknown_regional_features)
assert not unknown_regional_features

In [ ]:
dedup_cols = ['language', 'question', 'option_a', 'option_b', 'option_c', 'option_d']
n_before_dedup = len(df)
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
print(f'Rows before deduplication: {n_before_dedup}')
print(f'Rows after deduplication:  {len(df)}')

## Final exploration after cleaning

In [ ]:
print('Counts by language × regionality:')
display(pd.crosstab(df['language'], df['regionality']))

print('
Counts by language × source_split:')
display(pd.crosstab(df['language'], df['source_split']))

print('
Counts by domain:')
display(df['domain'].value_counts(dropna=False).sort_index().to_frame('n_examples'))

print('
Counts by language × domain:')
display(pd.crosstab(df['language'], df['domain']))

In [ ]:
plot_df = pd.crosstab(df['language'], df['regionality'])
ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('Clean INCLUDE examples per language and regionality')
ax.set_xlabel('Language')
ax.set_ylabel('Number of examples')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Convert to SFT-compatible records

In [ ]:
def safe_id_part(x):
    return str(x).strip().replace(' ', '_').replace('/', '_')


def make_record(row, idx):
    source_split = safe_id_part(row['source_split'])
    source_idx = int(row['source_idx'])
    language = row['language']
    return {
        'id': f'include_{language}_all_{source_split}_{source_idx}',
        'dataset': 'include',
        'language': language,
        'split': 'all',
        'task_type': 'mcq',
        'prompt': make_prompt(row),
        'answer': row['answer'],
        'metadata': {
            'country': clean_text(row.get('country', '')),
            'domain': clean_text(row.get('domain', '')),
            'subject': clean_text(row.get('subject', '')),
            'level': clean_text(row.get('level', '')),
            'regional_feature': clean_text(row.get('regional_feature', '')),
            'regionality': row['regionality'],
            'source_split': row['source_split'],
            'source_idx': source_idx,
        },
    }


records = [make_record(row, idx) for idx, row in df.iterrows()]
print(f'Built records: {len(records)}')
records[0]

## Save JSONL files

In [ ]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '
')


def read_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


for language in LANGUAGE_CONFIGS:
    language_records = [r for r in records if r['language'] == language]
    all_path = OUTPUT_DIR / f'include_{language}_all.jsonl'
    write_jsonl(language_records, all_path)
    print(f'Saved {len(language_records)} records to {all_path}')

    for regionality in ['regional', 'regional_implicit']:
        subset_records = [r for r in language_records if r['metadata']['regionality'] == regionality]
        path = OUTPUT_DIR / f'include_{language}_{regionality}.jsonl'
        write_jsonl(subset_records, path)
        print(f'Saved {len(subset_records)} records to {path}')

## Validate saved files

In [ ]:
def validate_record(record):
    assert record['dataset'] == 'include'
    assert record['language'] in LANGUAGE_CONFIGS
    assert record['split'] == 'all'
    assert record['task_type'] == 'mcq'
    assert record['answer'] in VALID_ANSWERS
    assert record['prompt'].strip()
    assert '

A) ' in record['prompt']
    assert record['metadata']['regionality'] in VALID_REGIONALITIES
    assert record['metadata']['source_split'] in SPLITS
    assert isinstance(record['metadata']['source_idx'], int)


for path in sorted(OUTPUT_DIR.glob('include_*.jsonl')):
    loaded = read_jsonl(path)
    ids = [r['id'] for r in loaded]
    prompts = [r['prompt'] for r in loaded]
    assert len(ids) == len(set(ids)), f'Duplicate ids in {path}'
    assert len(prompts) == len(set(prompts)), f'Duplicate prompts in {path}'
    for record in loaded:
        validate_record(record)
    print(f'Valid: {path.name} ({len(loaded)} records)')

## Final summary

In [ ]:
loaded_records = []
for language in LANGUAGE_CONFIGS:
    loaded_records.extend(read_jsonl(OUTPUT_DIR / f'include_{language}_all.jsonl'))

summary_df = pd.DataFrame({
    'language': r['language'],
    'answer': r['answer'],
    'regionality': r['metadata']['regionality'],
    'regional_feature': r['metadata']['regional_feature'],
    'domain': r['metadata']['domain'],
    'source_split': r['metadata']['source_split'],
} for r in loaded_records)

print('Final counts by language × regionality:')
display(pd.crosstab(summary_df['language'], summary_df['regionality']))

print('
Final counts by language × source_split:')
display(pd.crosstab(summary_df['language'], summary_df['source_split']))

print('
Final counts by domain:')
display(summary_df['domain'].value_counts().sort_index().to_frame('n_examples'))

print(f'Output directory: {OUTPUT_DIR}')